<a href="https://colab.research.google.com/github/vangogh1803/face_rec_final/blob/main/face_audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = "/content/drive/MyDrive/face_proj"


Mounted at /content/drive


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install -q datasets
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json


In [1]:
!pip install facenet-pytorch scikit-image tqdm opencv-python-headless


In [2]:
!kaggle datasets download -d hearfool/vggface2
!unzip vggface2.zip -d /content/vggface2

Streaming output truncated to the last 5000 lines.
  inflating: /content/vggface2/val/n001146/0284_01.jpg  
  inflating: /content/vggface2/val/n001146/0285_01.jpg  
  inflating: /content/vggface2/val/n001146/0286_01.jpg  
  inflating: /content/vggface2/val/n001146/0287_02.jpg  
  inflating: /content/vggface2/val/n001146/0288_01.jpg  
  inflating: /content/vggface2/val/n001146/0289_01.jpg  
  inflating: /content/vggface2/val/n001146/0290_01.jpg  
  inflating: /content/vggface2/val/n001146/0291_01.jpg  
  inflating: /content/vggface2/val/n001146/0292_01.jpg  
  inflating: /content/vggface2/val/n001146/0293_01.jpg  
  inflating: /content/vggface2/val/n001146/0294_02.jpg  
  inflating: /content/vggface2/val/n001146/0295_02.jpg  
  inflating: /content/vggface2/val/n001146/0296_01.jpg  
  inflating: /content/vggface2/val/n001146/0297_03.jpg  
  inflating: /content/vggface2/val/n001146/0298_01.jpg  
  inflating: /content/vggface2/val/n001146/0299_01.jpg  
  inflating: /content/vggface2/val/n0

In [ ]:
rm -rf /content/vgg_subset

In [ ]:
import gc
import torch

# 1. Delete your model and data variables if they exist
if 'backbone' in locals(): del backbone
if 'head' in locals(): del head
if 'train_loader' in locals(): del train_loader

# 2. Push Python to clear standard RAM
gc.collect()

# 3. Force PyTorch to dump its cache back to the GPU
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [ ]:
"""
Face Verification — InceptionResNetV1 (VGGFace2) + SubcenterArcFace
====================================================================
Target  : EER < 3% on VGGFace2 subset (1900 ids, ~20 imgs each)
Backbone: facenet-pytorch InceptionResNetV1, pretrained='vggface2'
Loss    : SubcenterArcFace  K=1 (i.e. standard ArcFace) for small sets
Eval    : Deterministic pairs + flip-TTA (built once, reused every epoch)

Key fixes over the 13.5% version
─────────────────────────────────
1. Backbone trains from epoch 0 with a low differential LR (1e-5).
   Freezing for 5 epochs wastes the pretrained features and forces the
   head to learn on a fixed representation that it will never see again.

2. Gallery split fixed: 20% gallery (was 45%). With only 20 imgs/id,
   45% gallery = 9 gallery + 11 train, which is fine for eval but hurts
   training badly. 20% = 4 gallery + 16 train is a better trade-off.

3. Deterministic eval pairs (seeded, built once). Random re-sampling
   caused EER to jump ±4% between epochs — best checkpoint was noise.

4. Flip-TTA in eval: embed(img) + embed(hflip(img)), then L2-norm.
   Free ~1-2% EER improvement on face data.

5. Removed `logits / 2` division — this was halving the effective
   scale from 48 to 24 and making the loss landscape very flat.

6. K=1 subcenters (standard ArcFace) for this dataset size.
   K=3 with only ~16 train imgs/class means each subcenter sees ~5
   samples — not enough to form meaningful intra-class clusters.
   Use K=1 now; if you scale to >500 imgs/class, bump to K=3.

7. Warmup-hold-cosine LR schedule (epoch-relative, resume-safe).
   CosineAnnealingWarmRestarts was resetting LR after unfreeze and
   spiking the backbone's gradients right when it started training.

8. Label smoothing moved to nn.CrossEntropyLoss (not the logit-halving
   hack). ε=0.05 (was 0.1) — less smoothing is better with ArcFace
   since the margin already acts as a regulariser.

9. AMP + gradient accumulation for OOM safety on 15GB GPUs.

10. torch.compile() opt-in for ~15% extra speed on PyTorch ≥ 2.0.
"""

import os, random, shutil, itertools, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from facenet_pytorch import InceptionResnetV1
from sklearn.metrics import roc_curve
from collections import defaultdict
from tqdm import tqdm
from PIL import Image

# ── CONFIG ───────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ALIGNED_DATA = "/content/vggface2/train"
SUBSET_PATH  = "/content/vgg_subset"

N_IDS        = 1900
N_IMGS       = 20
BATCH_SIZE   = 64           # effective 128 with ACC_STEPS=2
ACC_STEPS    = 2
EPOCHS       = 90
PATIENCE     = 20
IMG_SIZE     = 160          # InceptionResnetV1 native size
SEED         = 42

NUM_WORKERS  = 4
EVAL_START   = 2            # first eval at epoch 3 (0-indexed: 2)
EVAL_EVERY   = 2
USE_AMP      = True
USE_COMPILE  = False

# SubcenterArcFace
K_SUB        = 1            # 1 = standard ArcFace; bump to 3 if >200 imgs/id
ARC_S        = 48.0
ARC_M        = 0.35         # slightly softer than 0.4 for small sets

# LR schedule (warmup → hold → cosine decay)
BB_LR_PEAK   = 1e-5         # backbone — pretrained, keep conservative
HEAD_LR_PEAK = 3e-4
BB_LR_INIT   = 1e-7
HEAD_LR_INIT = 1e-7
WARMUP_EP    = 3
HOLD_EP      = 10

LABEL_SMOOTH = 0.05         # small; ArcFace margin already regularises

RESUME_FROM  = None         # e.g. "best_vgg.pt" to continue training

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# ── 1. SUBSET ────────────────────────────────────────────────────────
if not os.path.exists(SUBSET_PATH):
    ids = [d for d in os.listdir(ALIGNED_DATA)
           if os.path.isdir(os.path.join(ALIGNED_DATA, d))]
    selected = random.sample(ids, min(N_IDS, len(ids)))
    count = 0
    for ident in selected:
        src  = os.path.join(ALIGNED_DATA, ident)
        imgs = [f for f in os.listdir(src)
                if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
        if len(imgs) < 5:
            continue
        imgs = random.sample(imgs, min(N_IMGS, len(imgs)))
        dst  = os.path.join(SUBSET_PATH, ident)
        os.makedirs(dst, exist_ok=True)
        for f in imgs:
            shutil.copy(os.path.join(src, f), os.path.join(dst, f))
        count += 1
    print(f"✅ Subset: {count} identities")
else:
    print("Subset already exists.")

# ── 2. SPLIT ─────────────────────────────────────────────────────────
def get_samples(root):
    out = []
    for ident in os.listdir(root):
        p = os.path.join(root, ident)
        if not os.path.isdir(p): continue
        imgs = [os.path.join(p, f) for f in os.listdir(p)
                if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
        if len(imgs) >= 5:
            out.append((ident, imgs))
    return out

all_ids   = get_samples(SUBSET_PATH)
random.shuffle(all_ids)
n_classes = len(all_ids)
id_to_idx = {ident: i for i, (ident, _) in enumerate(all_ids)}

train_samples = []
gallery       = defaultdict(list)

for ident, imgs in all_ids:
    random.shuffle(imgs)
    # 20% gallery — keeps max images for training
    n_g = max(2, int(0.20 * len(imgs)))
    for p in imgs[n_g:]:  train_samples.append((p, id_to_idx[ident]))
    for p in imgs[:n_g]:  gallery[ident].append(p)

print(f"Classes: {n_classes} | Train: {len(train_samples)} | "
      f"Gallery: {sum(len(v) for v in gallery.values())}")

# ── 3. TRANSFORMS ────────────────────────────────────────────────────
train_tfm = transforms.Compose([
    transforms.Resize(180),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2,
                           saturation=0.1, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

val_tfm = transforms.Compose([
    transforms.Resize(180),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

# ── 4. DATASETS ──────────────────────────────────────────────────────
class TrainDataset(Dataset):
    def __init__(self, samples, tfm):
        self.samples = samples; self.tfm = tfm

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        try:    return self.tfm(Image.open(path).convert('RGB')), label
        except: return None, label

class GalleryDataset(Dataset):
    def __init__(self, gd, tfm):
        self.samples = [(p, ident) for ident, ps in gd.items() for p in ps]
        self.tfm = tfm

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        path, ident = self.samples[i]
        try:
            img = Image.open(path).convert('RGB')
            return (self.tfm(img),
                    self.tfm(transforms.functional.hflip(img)),
                    ident)
        except: return None, None, ident

def train_collate(batch):
    batch = [b for b in batch if b[0] is not None]
    if not batch: return None, None
    imgs, lbls = zip(*batch)
    return torch.stack(imgs), torch.tensor(lbls)

def gallery_collate(batch):
    batch = [b for b in batch if b[0] is not None]
    if not batch: return None, None, None
    i1, i2, lbls = zip(*batch)
    return torch.stack(i1), torch.stack(i2), list(lbls)

train_loader = DataLoader(
    TrainDataset(train_samples, train_tfm),
    batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, collate_fn=train_collate,
    pin_memory=True, persistent_workers=(NUM_WORKERS > 0))

gallery_loader = DataLoader(
    GalleryDataset(gallery, val_tfm),
    batch_size=128, num_workers=NUM_WORKERS,
    collate_fn=gallery_collate,
    pin_memory=True, persistent_workers=(NUM_WORKERS > 0))

# ── 5. DETERMINISTIC EVAL PAIRS ──────────────────────────────────────
def build_eval_pairs(gallery_dict, max_pairs=20000, seed=0):
    rng     = random.Random(seed)
    ordered = [(p, ident) for ident, ps in gallery_dict.items() for p in ps]
    id2idx  = defaultdict(list)
    for idx, (_, ident) in enumerate(ordered):
        id2idx[ident].append(idx)

    pos = []
    for v in id2idx.values():
        pos += list(itertools.combinations(v, 2))
    rng.shuffle(pos)
    pos = pos[:max_pairs]

    keys = list(id2idx.keys())
    neg  = []
    while len(neg) < len(pos):
        a, b = rng.sample(keys, 2)
        neg.append((rng.choice(id2idx[a]), rng.choice(id2idx[b])))

    I = torch.tensor([p[0] for p in pos + neg])
    J = torch.tensor([p[1] for p in pos + neg])
    T = np.array([1]*len(pos) + [0]*len(neg), dtype=np.int32)
    return I, J, T

print("Building deterministic eval pairs …")
EVAL_I, EVAL_J, EVAL_T = build_eval_pairs(gallery, max_pairs=20000)
print(f"Eval: {len(EVAL_T)//2} pos + {len(EVAL_T)//2} neg")

# ── 6. MODEL ─────────────────────────────────────────────────────────
class SubcenterArcFace(nn.Module):
    """
    K=1  →  standard ArcFace  (recommended for <200 imgs/class)
    K>1  →  subcenter variant  (for noisier / larger datasets)
    """
    def __init__(self, in_feat, n_classes, K=1, s=48.0, m=0.35):
        super().__init__()
        self.s = s; self.m = m; self.K = K
        self.weight = nn.Parameter(torch.FloatTensor(n_classes * K, in_feat))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, labels):
        cos_all = F.linear(F.normalize(x), F.normalize(self.weight))  # (B, C*K)
        if self.K > 1:
            cos = cos_all.view(x.size(0), -1, self.K).max(dim=2).values
        else:
            cos = cos_all
        cos = cos.clamp(-1 + 1e-7, 1 - 1e-7)

        theta   = torch.acos(cos)
        phi     = torch.cos(theta + self.m)
        one_hot = torch.zeros_like(cos).scatter_(1, labels.view(-1, 1), 1.)
        # No logit division — use full scale s
        return (one_hot * phi + (1 - one_hot) * cos) * self.s


# InceptionResnetV1 pretrained on VGGFace2 — already a face expert
backbone = InceptionResnetV1(pretrained='vggface2').to(device)
head     = SubcenterArcFace(512, n_classes, K=K_SUB,
                            s=ARC_S, m=ARC_M).to(device)

if USE_COMPILE and hasattr(torch, 'compile'):
    backbone = torch.compile(backbone)
    print("  ✅ torch.compile() applied.")

# Differential LR: backbone gets 30× lower LR than head
optimizer = optim.AdamW([
    {'params': backbone.parameters(), 'lr': BB_LR_INIT},
    {'params': head.parameters(),     'lr': HEAD_LR_INIT},
], weight_decay=5e-4)

ce_loss = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
scaler  = torch.cuda.amp.GradScaler(enabled=USE_AMP)

# ── 7. LR SCHEDULE (epoch-relative, resume-safe) ─────────────────────
def set_lr(epoch, start_epoch=0):
    """
    Warmup → hold → cosine, measured relative to start_epoch so that
    resuming mid-run doesn't reset the schedule back to warmup.
    """
    rel      = epoch - start_epoch
    remain   = EPOCHS - start_epoch
    decay_ep = remain - WARMUP_EP - HOLD_EP

    if rel < WARMUP_EP:
        f       = (rel + 1) / WARMUP_EP
        bb_lr   = BB_LR_INIT   + f * (BB_LR_PEAK   - BB_LR_INIT)
        head_lr = HEAD_LR_INIT + f * (HEAD_LR_PEAK - HEAD_LR_INIT)
    elif rel < WARMUP_EP + HOLD_EP:
        bb_lr, head_lr = BB_LR_PEAK, HEAD_LR_PEAK
    else:
        t       = (rel - WARMUP_EP - HOLD_EP) / max(1, decay_ep)
        factor  = max(0.01, 0.5 * (1 + math.cos(math.pi * t)))
        bb_lr   = BB_LR_PEAK   * factor
        head_lr = HEAD_LR_PEAK * factor

    optimizer.param_groups[0]['lr'] = bb_lr
    optimizer.param_groups[1]['lr'] = head_lr

# ── 8. RESUME ────────────────────────────────────────────────────────
start_epoch = 0
best_eer    = 1.0
if RESUME_FROM and os.path.isfile(RESUME_FROM):
    ckpt = torch.load(RESUME_FROM, map_location=device)
    backbone.load_state_dict(ckpt['backbone'])
    head.load_state_dict(ckpt['head'])
    start_epoch = ckpt.get('epoch', 0)
    best_eer    = ckpt.get('eer', 1.0)
    print(f"  ✅ Resumed from epoch {start_epoch}  (EER {best_eer*100:.2f}%)")
else:
    print("  Starting from scratch.")

# ── 9. EVALUATE ──────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(backbone):
    backbone.eval()
    feats = []
    for batch in gallery_loader:
        if batch[0] is None: continue
        i1, i2, _ = batch
        i1, i2 = i1.to(device), i2.to(device)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            f1 = F.normalize(backbone(i1), dim=1)
            f2 = F.normalize(backbone(i2), dim=1)
        # Flip-TTA: fuse normal + flipped embedding
        feats.append(F.normalize(f1 + f2, dim=1).cpu())
    if not feats: return 1.0
    feats  = torch.cat(feats)
    scores = (feats[EVAL_I] * feats[EVAL_J]).sum(1).numpy()
    fpr, tpr, _ = roc_curve(EVAL_T, scores)
    return float(fpr[np.nanargmin(np.abs(fpr - (1 - tpr)))])

# ── 10. TRAIN ────────────────────────────────────────────────────────
no_improve = 0

for epoch in range(start_epoch, EPOCHS):
    set_lr(epoch, start_epoch)

    backbone.train(); head.train()
    epoch_loss = 0.0
    n_batches  = 0
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc=f"Ep {epoch+1:03d}", leave=False)
    for i, (imgs, lbls) in enumerate(pbar):
        if imgs is None: continue

        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            emb    = F.normalize(backbone(imgs), dim=1)
            logits = head(emb, lbls)
            loss   = ce_loss(logits, lbls) / ACC_STEPS

        scaler.scale(loss).backward()
        epoch_loss += loss.item() * ACC_STEPS
        n_batches  += 1

        if (i + 1) % ACC_STEPS == 0 or (i + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(
                list(backbone.parameters()) + list(head.parameters()), 5.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        pbar.set_postfix(loss=f"{loss.item()*ACC_STEPS:.3f}")

    do_eval = (epoch >= EVAL_START) and (
        (epoch - EVAL_START) % EVAL_EVERY == 0 or epoch == EPOCHS - 1)

    eer     = evaluate(backbone) if do_eval else float('nan')
    bb_lr   = optimizer.param_groups[0]['lr']
    eer_str = f"{eer*100:5.2f}%" if not np.isnan(eer) else "  --  "
    avg     = epoch_loss / max(n_batches, 1)

    print(f"Epoch {epoch+1:3d} | Loss {avg:6.3f} | EER {eer_str} | "
          f"LR bb={bb_lr:.1e} hd={optimizer.param_groups[1]['lr']:.1e}")

    if not np.isnan(eer):
        if eer < best_eer:
            best_eer   = eer
            no_improve = 0
            torch.save({
                'epoch'   : epoch + 1,
                'backbone': backbone.state_dict(),
                'head'    : head.state_dict(),
                'eer'     : best_eer,
            }, "best_vgg.pt")
            print(f"  ✅ Saved  (EER {best_eer*100:.2f}%)")
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print("Early stopping.")
                break

print(f"\n🔥 FINAL BEST EER: {best_eer*100:.2f}%")

✅ Subset: 480 identities
Classes: 480 | Train: 7680 | Gallery: 1920
Building deterministic eval pairs …
Eval: 2880 pos + 2880 neg
  Starting from scratch.


Epoch   1 | Loss 23.172 | EER   --   | LR bb=3.4e-06 hd=1.0e-04


Epoch   2 | Loss 21.029 | EER   --   | LR bb=6.7e-06 hd=2.0e-04


Epoch   3 | Loss 18.201 | EER  3.85% | LR bb=1.0e-05 hd=3.0e-04
  ✅ Saved  (EER 3.85%)


Epoch   4 | Loss 15.756 | EER   --   | LR bb=1.0e-05 hd=3.0e-04


...

Epoch  61 | Loss  1.197 | EER  3.06% | LR bb=3.3e-06 hd=9.9e-05


Epoch  62 | Loss  1.191 | EER   --   | LR bb=3.1e-06 hd=9.3e-05


Epoch  63 | Loss  1.188 | EER  3.02% | LR bb=2.9e-06 hd=8.8e-05


Epoch  64 | Loss  1.195 | EER   --   | LR bb=2.7e-06 hd=8.2e-05


Epoch  65 | Loss  1.179 | EER  3.12% | LR bb=2.6e-06 hd=7.7e-05
Early stopping.

🔥 FINAL BEST EER: 2.64%


In [8]:
import os
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import random
import os, random, shutil, itertools, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from facenet_pytorch import InceptionResnetV1
from sklearn.metrics import roc_curve
from collections import defaultdict
from tqdm import tqdm
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATASET_PATH = "/content/vgg_subset"   # change if needed
ALIGNED_DATA = "/content/vggface2/train"

N_IDS        = 1900
N_IMGS       = 20
BATCH_SIZE   = 64           # effective 128 with ACC_STEPS=2
ACC_STEPS    = 2
EPOCHS       = 90
PATIENCE     = 20
IMG_SIZE     = 160          # InceptionResnetV1 native size
SEED         = 42

NUM_WORKERS  = 4
EVAL_START   = 2            # first eval at epoch 3 (0-indexed: 2)
EVAL_EVERY   = 2
USE_AMP      = True
USE_COMPILE  = False

# SubcenterArcFace
K_SUB        = 1            # 1 = standard ArcFace; bump to 3 if >200 imgs/id
ARC_S        = 48.0
ARC_M        = 0.35         # slightly softer than 0.4 for small sets

# LR schedule (warmup → hold → cosine decay)
BB_LR_PEAK   = 1e-5         # backbone — pretrained, keep conservative
HEAD_LR_PEAK = 3e-4
BB_LR_INIT   = 1e-7
HEAD_LR_INIT = 1e-7
WARMUP_EP    = 3
HOLD_EP      = 10

LABEL_SMOOTH = 0.05         # small; ArcFace margin already regularises

RESUME_FROM  = None         # e.g. "best_vgg.pt" to continue training

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
if not os.path.exists(DATASET_PATH):
    ids = [d for d in os.listdir(ALIGNED_DATA)
           if os.path.isdir(os.path.join(ALIGNED_DATA, d))]
    selected = random.sample(ids, min(N_IDS, len(ids)))
    count = 0
    for ident in selected:
        src  = os.path.join(ALIGNED_DATA, ident)
        imgs = [f for f in os.listdir(src)
                if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
        if len(imgs) < 5:
            continue
        imgs = random.sample(imgs, min(N_IMGS, len(imgs)))
        dst  = os.path.join(DATASET_PATH, ident)
        os.makedirs(dst, exist_ok=True)
        for f in imgs:
            shutil.copy(os.path.join(src, f), os.path.join(dst, f))
        count += 1
    print(f"✅ Subset: {count} identities")
else:
    print("Subset already exists.")

# -------- Dataset --------
class TrainDataset(Dataset):
    def __init__(self, root, tfm):
        self.samples = []
        self.tfm = tfm
        self.label_map = {}

        ids = os.listdir(root)

        for i, ident in enumerate(ids):
            path = os.path.join(root, ident)
            if not os.path.isdir(path):
                continue

            self.label_map[ident] = i

            for f in os.listdir(path):
                if f.lower().endswith(('.jpg', '.png', '.jpeg')):
                    self.samples.append((os.path.join(path, f), i))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.tfm(img), label


# -------- Transforms --------
transform = transforms.Compose([
    transforms.Resize(160),
    transforms.CenterCrop(160),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

# -------- Loader --------
dataset = TrainDataset(DATASET_PATH, transform)

train_loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2
)

print("✅ Dataset Loaded:", len(dataset))

✅ Subset: 480 identities
✅ Dataset Loaded: 9600


In [9]:
from facenet_pytorch import InceptionResnetV1
import torch.nn.functional as F

# Load pretrained face model
backbone = InceptionResnetV1(pretrained='vggface2').eval().to(device)

embeddings = []
labels = []

with torch.no_grad():
    for imgs, lbls in train_loader:
        imgs = imgs.to(device)

        emb = F.normalize(backbone(imgs), dim=1)

        embeddings.append(emb.cpu())
        labels.extend(lbls)

# Save results
torch.save({
    "embeddings": torch.cat(embeddings),
    "labels": labels
}, "face_results.pt")

print("🎉 Face embeddings saved as face_results.pt")

  0%|          | 0.00/107M [00:00<?, ?B/s]

🎉 Face embeddings saved as face_results.pt


In [13]:
!pip uninstall -y torch torchaudio

Found existing installation: torch 2.2.2
Uninstalling torch-2.2.2:
  Successfully uninstalled torch-2.2.2
Found existing installation: torchaudio 2.10.0+cpu
Uninstalling torchaudio-2.10.0+cpu:
  Successfully uninstalled torchaudio-2.10.0+cpu


In [1]:
!pip install torch torchaudio

In [2]:
import torchaudio

dataset = torchaudio.datasets.LIBRISPEECH(
    root="/content",
    url="train-clean-100",
    download=True
)

print("✅ LibriSpeech downloaded")

100%|██████████| 5.95G/5.95G [04:14<00:00, 25.1MB/s]


✅ LibriSpeech downloaded


In [50]:
import librosa
import os
import numpy as np

DATASET_PATH = "/content/audio_dataset"

clean_samples = []
clean_labels = []
label_map = {}

speakers = os.listdir(DATASET_PATH)

label_id = 0

for spk in speakers:
    spk_path = os.path.join(DATASET_PATH, spk)
    if not os.path.isdir(spk_path):
        continue

    label_map[spk] = label_id

    for file in os.listdir(spk_path):
        if not (file.endswith(".wav") or file.endswith(".flac")):
            continue

        path = os.path.join(spk_path, file)

        try:
            y, sr = librosa.load(path, sr=16000)

            # 🔥 FORCE NUMPY ARRAY
            if not isinstance(y, np.ndarray):
                continue

            if y.ndim != 1:
                continue

            if len(y) < 1000:
                continue

            clean_samples.append(path)
            clean_labels.append(label_id)

        except:
            continue

    label_id += 1

print("✅ Clean samples:", len(clean_samples))

✅ Clean samples: 1956


In [32]:
import os, shutil, random

SRC = "/content/LibriSpeech/train-clean-100"
DST = "/content/audio_dataset"

os.makedirs(DST, exist_ok=True)

N_SPEAKERS = 20   # 🔥 reduce if slow
N_SAMPLES = 25    # per speaker

speakers = os.listdir(SRC)
selected = random.sample(speakers, min(N_SPEAKERS, len(speakers)))

for spk in selected:
    spk_src = os.path.join(SRC, spk)
    spk_dst = os.path.join(DST, spk)
    os.makedirs(spk_dst, exist_ok=True)

    all_files = []

    for chapter in os.listdir(spk_src):
        ch_path = os.path.join(spk_src, chapter)
        for f in os.listdir(ch_path):
            if f.endswith(".flac"):
                all_files.append(os.path.join(ch_path, f))

    chosen = random.sample(all_files, min(N_SAMPLES, len(all_files)))

    for f in chosen:
        shutil.copy(f, spk_dst)

print("✅ Audio subset ready!")

✅ Audio subset ready!


In [33]:
import torchaudio.transforms as T

mfcc_transform = T.MFCC(
    sample_rate=16000,
    n_mfcc=40
)

/usr/local/lib/python3.12/dist-packages/torchaudio/functional/functional.py:581: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (128) may be set too high. Or, the value for `n_freqs` (201) may be set too low.
  warnings.warn(


In [34]:
import torchaudio

bundle = torchaudio.pipelines.WAV2VEC2_BASE
wav2vec = bundle.get_model()
wav2vec.eval()

print("✅ wav2vec loaded")

✅ wav2vec loaded


In [60]:
import librosa
import os

DATASET_PATH = "/content/audio_dataset"

clean_samples = []
clean_labels = []
label_map = {}

speakers = os.listdir(DATASET_PATH)

label_id = 0

for spk in speakers:
    spk_path = os.path.join(DATASET_PATH, spk)
    if not os.path.isdir(spk_path):
        continue

    label_map[spk] = label_id

    for file in os.listdir(spk_path):
        if not (file.endswith(".wav") or file.endswith(".flac")):
            continue

        path = os.path.join(spk_path, file)

        try:
            y, sr = librosa.load(path, sr=16000)

            # keep only valid audio
            if y is not None and len(y) > 1000:
                clean_samples.append(path)
                clean_labels.append(label_id)

        except:
            continue

    label_id += 1

print("✅ Clean samples:", len(clean_samples))

✅ Clean samples: 1956


In [71]:
import torch
from torch.utils.data import Dataset
import librosa
import numpy as np

class SafeAudioDataset(Dataset):
    def __init__(self, samples, labels):
        self.samples = samples
        self.labels = labels

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path = self.samples[idx]

        y, sr = librosa.load(path, sr=16000)

        # force numpy array
        y = np.asarray(y)

        # ensure 1D
        if y.ndim != 1:
            y = y.flatten()

        waveform = torch.tensor(y, dtype=torch.float32)

        max_len = sr * 3

        # 🔥 IMPORTANT: use numel(), NOT shape
        if waveform.numel() > max_len:
            waveform = waveform[:max_len]
        else:
            pad = max_len - waveform.numel()
            waveform = torch.nn.functional.pad(waveform, (0, pad))

        return waveform, self.labels[idx]

In [72]:
from torch.utils.data import DataLoader

dataset = SafeAudioDataset(clean_samples, clean_labels)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

In [68]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset, DataLoader
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# DATASET
# -------------------------
class AudioDataset(Dataset):
    def __init__(self, root):
        self.samples = []
        self.labels = []
        self.label_map = {}

        speakers = os.listdir(root)

        for i, spk in enumerate(speakers):
            spk_path = os.path.join(root, spk)
            if not os.path.isdir(spk_path):
                continue

            self.label_map[spk] = i

            for file in os.listdir(spk_path):
                if file.endswith(".wav") or file.endswith(".flac"):
                    self.samples.append(os.path.join(spk_path, file))
                    self.labels.append(i)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        waveform, sr = load_audio(self.samples[idx])
        waveform = waveform.mean(dim=0)

        max_len = sr * 3
        if waveform.shape[0] > max_len:
            waveform = waveform[:max_len]
        else:
            waveform = torch.nn.functional.pad(waveform, (0, max_len - waveform.shape[0]))

        return waveform, self.labels[idx]


DATASET_PATH = "/content/audio_dataset"
dataset = AudioDataset(DATASET_PATH)
loader = DataLoader(dataset, batch_size=16, shuffle=False)

print("✅ Dataset loaded:", len(dataset))

# -------------------------
# MFCC
# -------------------------
import torchaudio.transforms as T

mfcc_transform = T.MFCC(
    sample_rate=16000,
    n_mfcc=40
).to(device)

# -------------------------
# MODEL
# -------------------------
class SimpleAudioModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(40, 128)
        self.fc2 = nn.Linear(128, 128)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        mfcc = mfcc_transform(x)
        mfcc = mfcc.mean(dim=2)

        emb = torch.relu(self.fc1(mfcc))
        emb = F.normalize(self.fc2(emb), dim=1)

        logits = self.classifier(emb)

        return emb, logits


model = SimpleAudioModel(num_classes=len(dataset.label_map)).to(device)

print("✅ Model rebuilt")

✅ Dataset loaded: 1956
✅ Model rebuilt


In [74]:
model.eval()

audio_embeddings = []
audio_labels = []

with torch.no_grad():
    for waveform, label in loader:
        # move to device
        waveform = waveform.to(device)
        label = label.to(device)   # ✅ FIX 1 (important)

        # ensure correct shape (B, T)
        if waveform.dim() == 1:
            waveform = waveform.unsqueeze(0)   # ✅ FIX 2 (safety)

        # forward pass
        emb, _ = model(waveform)

        # normalize embeddings (VERY IMPORTANT for similarity)
        emb = torch.nn.functional.normalize(emb, dim=1)   # ✅ FIX 3

        audio_embeddings.append(emb.cpu())
        audio_labels.extend(label.cpu().numpy())   # ✅ FIX 4

# save safely
torch.save({
    "embeddings": torch.cat(audio_embeddings, dim=0),
    "labels": audio_labels
}, "audio_results.pt")

print("✅ Regenerated audio_results.pt")

✅ Regenerated audio_results.pt


In [79]:
import torch.nn as nn
import torch.nn.functional as F
import torchaudio.transforms as T

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

mfcc_transform = T.MFCC(
    sample_rate=16000,
    n_mfcc=40
).to(device)


class SimpleAudioModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(40, 128)
        self.fc2 = nn.Linear(128, 128)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        mfcc = mfcc_transform(x)
        mfcc = mfcc.mean(dim=2)

        emb = torch.relu(self.fc1(mfcc))
        emb = F.normalize(self.fc2(emb), dim=1)

        logits = self.classifier(emb)

        return emb, logits


model = SimpleAudioModel(num_classes=len(set(clean_labels))).to(device)

In [80]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    model.train()
    total_loss = 0

    for waveform, label in loader:
        waveform = waveform.to(device)
        label = label.to(device)

        emb, logits = model(waveform)

        loss = criterion(logits, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.3f}")

Epoch 1, Loss: 525.637
Epoch 2, Loss: 485.132
Epoch 3, Loss: 445.330
Epoch 4, Loss: 410.145
Epoch 5, Loss: 377.038
Epoch 6, Loss: 345.911
Epoch 7, Loss: 315.700
Epoch 8, Loss: 287.787
Epoch 9, Loss: 261.808
Epoch 10, Loss: 236.807


In [85]:
import torch.nn.functional as F
import torch

model.eval()

audio_embeddings = []
audio_labels = []

with torch.no_grad():
    for waveform, label in loader:
        waveform = waveform.to(device)
        label = label.to(device)

        # normalize input (same as training)
        waveform = waveform / (waveform.abs().max(dim=1, keepdim=True)[0] + 1e-6)

        emb, _ = model(waveform)

        # normalize embeddings (VERY IMPORTANT)
        emb = F.normalize(emb, dim=1)

        audio_embeddings.append(emb.cpu())
        audio_labels.extend(label.cpu().numpy())

# combine everything
audio_embeddings = torch.cat(audio_embeddings, dim=0)

# save locally
torch.save({
    "embeddings": audio_embeddings,
    "labels": audio_labels
}, "audio_results.pt")

print("✅ Saved at /content/audio_results.pt")

✅ Saved at /content/audio_results.pt


In [88]:
import shutil

shutil.copy("audio_results.pt", "/content/drive/MyDrive/audio_results.pt")

print("🔥 Saved permanently to Drive")

🔥 Saved permanently to Drive


In [91]:
import numpy as np
from sklearn.metrics import roc_curve
import torch.nn.functional as F

audio = torch.load("audio_results.pt")

audio_emb = F.normalize(audio["embeddings"], dim=1)
labels = np.array(audio["labels"])

pairs = []
pair_labels = []

N = len(audio_emb)
MAX_PAIRS = 10000

count = 0

for i in range(N):
    for j in range(i+1, N):
        if count >= MAX_PAIRS:
            break

        sim = torch.dot(audio_emb[i], audio_emb[j]).item()
        pairs.append(sim)

        if labels[i] == labels[j]:
            pair_labels.append(1)
        else:
            pair_labels.append(0)

        count += 1
    if count >= MAX_PAIRS:
        break

pairs = np.array(pairs)
pair_labels = np.array(pair_labels)

# ROC
fpr, tpr, thresholds = roc_curve(pair_labels, pairs)
frr = 1 - tpr

eer_idx = np.nanargmin(np.abs(fpr - frr))
eer = fpr[eer_idx]

print(f"🎧 Audio EER: {eer*100:.2f}%")

# FAR / FRR
th = thresholds[eer_idx]

far = np.mean((pairs >= th) & (pair_labels == 0))
frr_val = np.mean((pairs < th) & (pair_labels == 1))

print(f"FAR: {far*100:.2f}%")
print(f"FRR: {frr_val*100:.2f}%")

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy.core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy.core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy.core.multiarray.scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn mor

In [90]:
import torch
import numpy as np
from sklearn.metrics import roc_curve

face = torch.load("/content/drive/MyDrive/face_results.pt", weights_only=False)
audio = torch.load("/content/drive/MyDrive/audio_results.pt", weights_only=False)

face_emb = torch.nn.functional.normalize(face["embeddings"], dim=1)
audio_emb = torch.nn.functional.normalize(audio["embeddings"], dim=1)

labels = np.array(face["labels"])

# Match size
N = min(len(face_emb), len(audio_emb))
face_emb = face_emb[:N]
audio_emb = audio_emb[:N]
labels = labels[:N]

pairs = []
pair_labels = []

import random

MAX_PAIRS = 10000

for _ in range(MAX_PAIRS):
    i, j = random.sample(range(N), 2)

    face_sim = torch.dot(face_emb[i], face_emb[j]).item()
    audio_sim = torch.dot(audio_emb[i], audio_emb[j]).item()

    # 🔥 weighted fusion
    combined_sim = 0.7 * face_sim + 0.3 * audio_sim

    pairs.append(combined_sim)

    if labels[i] == labels[j]:
        pair_labels.append(1)
    else:
        pair_labels.append(0)

pairs = np.array(pairs)
pair_labels = np.array(pair_labels)

# ROC
fpr, tpr, thresholds = roc_curve(pair_labels, pairs)
frr = 1 - tpr

eer_idx = np.nanargmin(np.abs(fpr - frr))
eer = fpr[eer_idx]

print(f"🔥 Fusion EER: {eer*100:.2f}%")

🔥 Fusion EER: 3.78%
